# Exercise 2: Multi-Agent Systems with LangGraph

In this exercise, you will learn to:
1. Build a supervisor pattern with multiple specialized agents
2. Implement agent-to-agent communication
3. Create a collaborative multi-agent workflow
4. Handle shared state across agents

**Prerequisites**: Complete Exercise 1 (Advanced Agents) first.

## Setup

In [ ]:
# Install required packages
!pip3 install langchain langchain-openai langgraph --quiet

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify that the required environment variables are set
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your .env file."
assert os.environ.get("AZURE_OPENAI_DEPLOYMENT"), "AZURE_OPENAI_DEPLOYMENT is not set! Check your .env file."

print("Environment variables loaded successfully!")

In [ ]:
from typing import Annotated, TypedDict, Literal, Sequence
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent, ToolNode
import json
import operator

# Initialize the LLM
llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model=os.environ["AZURE_OPENAI_DEPLOYMENT"],api_key=os.environ["OPENAI_API_KEY"])

## Part 1: Understanding Multi-Agent Architectures

Multi-agent systems allow you to:
- **Specialize**: Each agent focuses on one domain
- **Scale**: Handle complex tasks by dividing work
- **Collaborate**: Agents can build on each other's work

In [ ]:
# Let's visualize the supervisor pattern we'll build
print("""
┌─────────────────────────────────────────────────────────────┐
│                    SUPERVISOR PATTERN                       │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│                     ┌──────────────┐                        │
│      User ─────────▶│  SUPERVISOR  │                        │
│                     │   (Router)   │                        │
│                     └──────┬───────┘                        │
│                            │                                │
│         ┌──────────────────┼──────────────────┐             │
│         ▼                  ▼                  ▼             │
│   ┌───────────┐     ┌───────────┐     ┌───────────┐        │
│   │ RESEARCHER│     │   CODER   │     │  WRITER   │        │
│   │   Agent   │     │   Agent   │     │   Agent   │        │
│   └───────────┘     └───────────┘     └───────────┘        │
│                                                             │
└─────────────────────────────────────────────────────────────┘

The supervisor decides which agent should handle each task,
then aggregates results and continues until the task is complete.
""")

## Part 2: Building the Supervisor Pattern

We'll create a research team with:
- **Supervisor**: Routes tasks to the right agent
- **Researcher**: Finds information
- **Coder**: Writes and analyzes code
- **Writer**: Creates written content

In [ ]:
# Define tools for each specialized agent

# Researcher tools
@tool
def web_search(query: str) -> str:
    """Search the web for information on a topic."""
    # Simulated search results
    results = {
        "python": "Python is a high-level programming language known for readability. Latest version: 3.12. Popular for AI/ML.",
        "langchain": "LangChain is a framework for building LLM applications. Key features: chains, agents, memory, tools.",
        "react": "React is a JavaScript library for building user interfaces. Developed by Facebook. Uses virtual DOM.",
        "default": f"Search results for '{query}': Found information about the topic including key facts and recent developments."
    }
    for key in results:
        if key in query.lower():
            return results[key]
    return results["default"]

@tool
def get_facts(topic: str) -> str:
    """Get factual information about a topic."""
    return f"Key facts about {topic}: This is a significant topic with multiple aspects worth exploring."

# Coder tools
@tool
def write_code(language: str, task: str) -> str:
    """Write code in a specified language for a given task."""
    code_examples = {
        "python": f'''```python
# Solution for: {task}
def solution():
    # Implementation here
    result = "Task completed"
    return result

if __name__ == "__main__":
    print(solution())
```''',
        "javascript": f'''```javascript
// Solution for: {task}
function solution() {{
    const result = "Task completed";
    return result;
}}
console.log(solution());
```'''
    }
    return code_examples.get(language.lower(), f"```{language}\n// Code for {task}\n```")

@tool
def analyze_code(code: str) -> str:
    """Analyze code for potential issues and improvements."""
    return f"Code analysis: The code appears well-structured. Suggestions: Add error handling, consider edge cases, add documentation."

# Writer tools
@tool
def write_document(title: str, content_type: str, key_points: str) -> str:
    """Write a document with specified title, type (blog, report, summary), and key points."""
    return f"""# {title}

## Overview
This {content_type} covers the following key points:

{key_points}

## Conclusion
In summary, these points highlight the importance of understanding {title.lower()}.
"""

@tool
def summarize(text: str) -> str:
    """Summarize a piece of text."""
    words = text.split()
    summary_length = min(30, len(words) // 2)
    return f"Summary: {' '.join(words[:summary_length])}..."

print("Tools defined for all agents!")

In [ ]:
# Define the multi-agent state
class MultiAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str
    task_completed: bool
    research_findings: str
    code_output: str
    written_content: str

# Agent names
AGENTS = ["researcher", "coder", "writer"]

# Supervisor system prompt
SUPERVISOR_PROMPT = """You are a supervisor managing a team of agents: {agents}.

Your job is to:
1. Analyze the user's request
2. Decide which agent should handle it (or if it's complete)
3. Route to the appropriate agent

Agent capabilities:
- researcher: Searches for information, gathers facts
- coder: Writes and analyzes code
- writer: Creates documents, summaries, blog posts

Respond with ONLY the agent name or 'FINISH' if the task is complete.
Agent names: researcher, coder, writer, FINISH

Current task progress:
- Research findings: {research}
- Code output: {code}
- Written content: {writing}
"""

In [ ]:
# Create the supervisor node
def supervisor_node(state: MultiAgentState) -> dict:
    """The supervisor decides which agent should act next."""
    messages = state["messages"]
    
    # Format the supervisor prompt with current state
    prompt = SUPERVISOR_PROMPT.format(
        agents=", ".join(AGENTS),
        research=state.get("research_findings", "None yet"),
        code=state.get("code_output", "None yet"),
        writing=state.get("written_content", "None yet")
    )
    
    response = llm.invoke([
        SystemMessage(content=prompt),
        *messages
    ])
    
    # Parse the response to get the next agent
    next_agent = response.content.strip().lower()
    
    # Validate the response
    if next_agent not in AGENTS and next_agent != "finish":
        # Default to finish if invalid response
        next_agent = "finish"
    
    print(f"\n🎯 Supervisor decision: Route to '{next_agent}'")
    
    return {
        "next_agent": next_agent,
        "task_completed": next_agent == "finish"
    }

def route_to_agent(state: MultiAgentState) -> str:
    """Route based on supervisor's decision."""
    next_agent = state["next_agent"]
    if next_agent == "finish":
        return "finish"
    return next_agent

In [ ]:
# Create agent nodes
researcher_tools = [web_search, get_facts]
coder_tools = [write_code, analyze_code]
writer_tools = [write_document, summarize]

def create_agent_node(name: str, tools: list, system_prompt: str):
    """Factory function to create agent nodes."""
    agent_llm = llm.bind_tools(tools)
    tool_node = ToolNode(tools)
    
    def agent_node(state: MultiAgentState) -> dict:
        """Execute the agent's task."""
        messages = state["messages"]
        
        print(f"\n🤖 {name.upper()} Agent working...")
        
        # Create agent-specific context
        agent_messages = [
            SystemMessage(content=system_prompt),
            *messages
        ]
        
        # Get agent's response
        response = agent_llm.invoke(agent_messages)
        
        # Execute any tool calls
        result_content = response.content
        if hasattr(response, 'tool_calls') and response.tool_calls:
            for tc in response.tool_calls:
                tool_name = tc['name']
                tool_args = tc['args']
                # Find and execute the tool
                for t in tools:
                    if t.name == tool_name:
                        tool_result = t.invoke(tool_args)
                        result_content = f"{response.content}\n\nTool '{tool_name}' result:\n{tool_result}"
                        print(f"   🔧 Used tool: {tool_name}")
        
        # Create the agent's message
        agent_message = AIMessage(content=f"[{name.upper()}]: {result_content}")
        
        # Update the appropriate state field
        updates = {"messages": [agent_message]}
        if name == "researcher":
            updates["research_findings"] = result_content
        elif name == "coder":
            updates["code_output"] = result_content
        elif name == "writer":
            updates["written_content"] = result_content
        
        return updates
    
    return agent_node

# Create the agent nodes
researcher_node = create_agent_node(
    "researcher",
    researcher_tools,
    "You are a research agent. Find information and facts relevant to the user's request. Be thorough and cite your sources."
)

coder_node = create_agent_node(
    "coder",
    coder_tools,
    "You are a coding agent. Write clean, well-documented code. If given existing code, analyze it and suggest improvements."
)

writer_node = create_agent_node(
    "writer",
    writer_tools,
    "You are a writing agent. Create clear, engaging content. Use the research and code outputs if available."
)

def finish_node(state: MultiAgentState) -> dict:
    """Final node that compiles the results."""
    print("\n✅ Task completed!")
    
    summary = ["\n=== FINAL OUTPUT ==="]
    if state.get("research_findings"):
        summary.append(f"\n📚 Research:\n{state['research_findings'][:200]}...")
    if state.get("code_output"):
        summary.append(f"\n💻 Code:\n{state['code_output'][:200]}...")
    if state.get("written_content"):
        summary.append(f"\n📝 Writing:\n{state['written_content'][:200]}...")
    
    final_message = AIMessage(content="\n".join(summary))
    return {"messages": [final_message]}

print("Agent nodes created!")

In [ ]:
# Build the multi-agent graph
multi_agent_graph = StateGraph(MultiAgentState)

# Add all nodes
multi_agent_graph.add_node("supervisor", supervisor_node)
multi_agent_graph.add_node("researcher", researcher_node)
multi_agent_graph.add_node("coder", coder_node)
multi_agent_graph.add_node("writer", writer_node)
multi_agent_graph.add_node("finish", finish_node)

# Add edges
multi_agent_graph.add_edge(START, "supervisor")

# Conditional edges from supervisor
multi_agent_graph.add_conditional_edges(
    "supervisor",
    route_to_agent,
    {
        "researcher": "researcher",
        "coder": "coder",
        "writer": "writer",
        "finish": "finish"
    }
)

# After each agent, go back to supervisor
for agent in AGENTS:
    multi_agent_graph.add_edge(agent, "supervisor")

# Finish goes to END
multi_agent_graph.add_edge("finish", END)

# Compile
multi_agent_app = multi_agent_graph.compile()

print("Multi-agent graph compiled!")

In [ ]:
# Visualize the graph
try:
    from IPython.display import Image, display
    display(Image(multi_agent_app.get_graph().draw_mermaid_png()))
except:
    print(multi_agent_app.get_graph().draw_ascii())

In [ ]:
# Test the multi-agent system!
def run_multi_agent(query: str):
    """Run the multi-agent system with a query."""
    print(f"\n{'='*60}")
    print(f"📋 TASK: {query}")
    print('='*60)
    
    initial_state = {
        "messages": [HumanMessage(content=query)],
        "next_agent": "",
        "task_completed": False,
        "research_findings": "",
        "code_output": "",
        "written_content": ""
    }
    
    result = multi_agent_app.invoke(initial_state)
    
    print("\n" + "="*60)
    print("📊 FINAL RESULT")
    print("="*60)
    print(result["messages"][-1].content)
    
    return result

# Test with a research task
run_multi_agent("Research Python's latest features and write a short summary.")

In [ ]:
# Test with a coding task
run_multi_agent("Write a Python function to calculate fibonacci numbers and explain how it works.")

In [ ]:
# Test with a complex multi-step task
run_multi_agent(
    "Research LangChain, then write Python code that demonstrates its key features, "
    "and finally write a blog post explaining the code."
)

## Part 3: Collaborative Pattern (Agent Debate)

Let's implement a collaborative pattern where agents iterate on a shared artifact.

In [ ]:
# State for collaborative agents
class CollaborativeState(TypedDict):
    messages: Annotated[list, add_messages]
    draft: str
    critiques: list[str]
    iteration: int
    max_iterations: int
    is_approved: bool

# Writer agent creates/revises content
def collaborative_writer(state: CollaborativeState) -> dict:
    """Write or revise the draft based on critiques."""
    iteration = state.get("iteration", 0) + 1
    current_draft = state.get("draft", "")
    critiques = state.get("critiques", [])
    
    print(f"\n✍️  Writer (Iteration {iteration})")
    
    if not current_draft:
        # First draft
        prompt = f"""Write a first draft based on this request:
{state['messages'][0].content}

Create a clear, well-structured piece of writing."""
    else:
        # Revision
        prompt = f"""Revise this draft based on the critiques:

CURRENT DRAFT:
{current_draft}

CRITIQUES:
{chr(10).join(critiques)}

Create an improved version addressing the feedback."""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    new_draft = response.content
    
    print(f"   Draft updated ({len(new_draft)} chars)")
    
    return {
        "draft": new_draft,
        "iteration": iteration,
        "messages": [AIMessage(content=f"[WRITER] Draft v{iteration} created.")]
    }

# Critic agent reviews and critiques
def collaborative_critic(state: CollaborativeState) -> dict:
    """Critique the current draft."""
    draft = state["draft"]
    iteration = state["iteration"]
    
    print(f"\n🔍 Critic reviewing...")
    
    prompt = f"""Review this draft and provide constructive criticism:

DRAFT:
{draft}

Provide specific, actionable feedback. If the draft is excellent and needs no changes, 
respond with exactly: APPROVED

Otherwise, list 2-3 specific improvements needed."""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    critique = response.content
    
    is_approved = "APPROVED" in critique.upper()
    
    if is_approved:
        print("   ✅ Approved!")
    else:
        print(f"   📝 Feedback provided")
    
    return {
        "critiques": [critique],
        "is_approved": is_approved,
        "messages": [AIMessage(content=f"[CRITIC] Review complete. Approved: {is_approved}")]
    }

def should_continue_collaboration(state: CollaborativeState) -> str:
    """Check if we should continue iterating."""
    if state.get("is_approved"):
        return "end"
    if state.get("iteration", 0) >= state.get("max_iterations", 3):
        return "end"
    return "writer"

In [ ]:
# Build the collaborative graph
collab_graph = StateGraph(CollaborativeState)

collab_graph.add_node("writer", collaborative_writer)
collab_graph.add_node("critic", collaborative_critic)

collab_graph.add_edge(START, "writer")
collab_graph.add_edge("writer", "critic")
collab_graph.add_conditional_edges(
    "critic",
    should_continue_collaboration,
    {"writer": "writer", "end": END}
)

collab_app = collab_graph.compile()
print("Collaborative graph compiled!")

In [ ]:
# Test the collaborative system
def run_collaborative(task: str, max_iterations: int = 3):
    """Run the collaborative writer-critic system."""
    print(f"\n{'='*60}")
    print(f"📋 COLLABORATIVE TASK: {task}")
    print(f"   Max iterations: {max_iterations}")
    print('='*60)
    
    initial_state = {
        "messages": [HumanMessage(content=task)],
        "draft": "",
        "critiques": [],
        "iteration": 0,
        "max_iterations": max_iterations,
        "is_approved": False
    }
    
    result = collab_app.invoke(initial_state)
    
    print("\n" + "="*60)
    print("📊 FINAL DRAFT")
    print("="*60)
    print(f"Iterations: {result['iteration']}")
    print(f"Approved: {result['is_approved']}")
    print("\n" + result["draft"])
    
    return result

# Test with a writing task
run_collaborative("Write a short paragraph explaining why Python is popular for data science.")

## 🎯 Challenge Exercises

Now it's your turn!

### Challenge 1: Add a New Agent

Add a "Fact Checker" agent to the supervisor pattern that verifies claims made by other agents.

In [ ]:
# TODO: Implement a fact_checker agent
# 1. Create fact-checking tools
# 2. Create the fact_checker node
# 3. Add it to the graph
# 4. Update the supervisor prompt to include it

@tool
def verify_claim(claim: str) -> str:
    """Verify if a claim is accurate."""
    # Your implementation here
    pass

# Your implementation here

### Challenge 2: Parallel Agent Execution

Modify the system so that the researcher and coder can work in parallel when their tasks are independent.

In [ ]:
# TODO: Implement parallel execution
# Hint: Use conditional edges that fan out to multiple agents
# and a join node that waits for all to complete

# Your implementation here

### Challenge 3: Agent Memory Sharing

Create a shared "blackboard" that all agents can read from and write to.

In [ ]:
# TODO: Implement a shared blackboard pattern
# 1. Add a 'blackboard' field to the state (dict)
# 2. Each agent can read/write to specific keys
# 3. Agents can see what other agents have contributed

class BlackboardState(TypedDict):
    messages: Annotated[list, add_messages]
    blackboard: dict  # Shared knowledge base
    # Add more fields as needed

# Your implementation here

## Summary

In this exercise, you learned:

1. **Supervisor Pattern**: A central agent routes tasks to specialists
2. **Agent Specialization**: Each agent has its own tools and expertise
3. **Collaborative Pattern**: Agents iterate on shared artifacts
4. **State Sharing**: How agents communicate through shared state

These patterns enable building sophisticated multi-agent systems for complex tasks!